## config.py

In [1]:
# config.py

import albumentations as A
import cv2
import torch

from albumentations.pytorch import ToTensorV2

#DATASET = '/kaggle/input/pascal-voc-yolo-works-with-albumentations/PASCAL_VOC'
DATASET = 'NYUv2'
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# seed_everything()  # If you want deterministic behavior
NUM_WORKERS = 0
BATCH_SIZE = 8
IMAGE_SIZE = 416
# Moved NUM_CLASSES to the end of the file to get the length of the classes list
LEARNING_RATE = 1e-5
WEIGHT_DECAY = 1e-4
NUM_EPOCHS = 100
CONF_THRESHOLD = 0.05
MAP_IOU_THRESH = 0.5
NMS_IOU_THRESH = 0.45
S = [IMAGE_SIZE // 32, IMAGE_SIZE // 16, IMAGE_SIZE // 8]
PIN_MEMORY = True
LOAD_MODEL = False
SAVE_MODEL = True
CHECKPOINT_FILE = "checkpoint.pth.tar"
IMG_DIR = DATASET + "/images/"
LABEL_DIR = DATASET + "/labels/"
NYU_PATH = "./resources/nyu_depth_v2_labeled.mat"

'''
In YoLov3, there are three prediction scale, where each predication scale has three anchor boxes. This means that
when the entire image is divided into cells, each cell of the image has three anchor boxes. For example, the first 
prediction scale has the image divided into grid cells of dimensions 13x13, and each cell of this prediction scale has 
three anchor boxes.
Each tuple consists of the height and width of the respective anchor box. 
Each list grouping consists of the dimensions (height and width) of the anchor boxes for that scale prediction level. 

Note: The dimensions have been scaled to be relative to the image. 
'''
ANCHORS = [
    # Largest anchor boxes -> used for prediction on the coarsest grid, where predicting larger anchor boxes is easier.
    [(0.28, 0.22), (0.38, 0.48), (0.9, 0.78)],
    [(0.07, 0.15), (0.15, 0.11), (0.14, 0.29)],
    [(0.02, 0.03), (0.04, 0.07), (0.08, 0.06)],
]

scale = 1.1
train_transforms = A.Compose(
    [
        A.LongestMaxSize(max_size=int(IMAGE_SIZE * scale)),
        A.PadIfNeeded(
            min_height=int(IMAGE_SIZE * scale),
            min_width=int(IMAGE_SIZE * scale),
            border_mode=cv2.BORDER_CONSTANT,
            value=0
        ),
        A.RandomCrop(width=IMAGE_SIZE, height=IMAGE_SIZE),
        A.ColorJitter(brightness=0.6, contrast=0.6,
                      saturation=0.6, hue=0.6, p=0.4),
        A.HorizontalFlip(p=0.5),
        A.Blur(p=0.1),
        A.CLAHE(p=0.1),
        A.Posterize(p=0.1),
        A.ToGray(p=0.1),
        A.ChannelShuffle(p=0.05),
        A.Normalize(mean=[0, 0, 0], std=[1, 1, 1], max_pixel_value=255, ),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(
        format="yolo", min_visibility=0.4, label_fields=[], ),
)
test_transforms = A.Compose(
    [
        A.LongestMaxSize(max_size=IMAGE_SIZE),
        A.PadIfNeeded(
            min_height=IMAGE_SIZE, min_width=IMAGE_SIZE, border_mode=cv2.BORDER_CONSTANT, value=0
        ),
        A.Normalize(mean=[0, 0, 0], std=[1, 1, 1], max_pixel_value=255, ),
        ToTensorV2(),
    ],
    bbox_params=A.BboxParams(
        format="yolo", min_visibility=0.4, label_fields=[]),
)

PASCAL_CLASSES = [
    "aeroplane",
    "bicycle",
    "bird",
    "boat",
    "bottle",
    "bus",
    "car",
    "cat",
    "chair",
    "cow",
    "diningtable",
    "dog",
    "horse",
    "motorbike",
    "person",
    "pottedplant",
    "sheep",
    "sofa",
    "train",
    "tvmonitor"
]

COCO_LABELS = ['person',
               'bicycle',
               'car',
               'motorcycle',
               'airplane',
               'bus',
               'train',
               'truck',
               'boat',
               'traffic light',
               'fire hydrant',
               'stop sign',
               'parking meter',
               'bench',
               'bird',
               'cat',
               'dog',
               'horse',
               'sheep',
               'cow',
               'elephant',
               'bear',
               'zebra',
               'giraffe',
               'backpack',
               'umbrella',
               'handbag',
               'tie',
               'suitcase',
               'frisbee',
               'skis',
               'snowboard',
               'sports ball',
               'kite',
               'baseball bat',
               'baseball glove',
               'skateboard',
               'surfboard',
               'tennis racket',
               'bottle',
               'wine glass',
               'cup',
               'fork',
               'knife',
               'spoon',
               'bowl',
               'banana',
               'apple',
               'sandwich',
               'orange',
               'broccoli',
               'carrot',
               'hot dog',
               'pizza',
               'donut',
               'cake',
               'chair',
               'couch',
               'potted plant',
               'bed',
               'dining table',
               'toilet',
               'tv',
               'laptop',
               'mouse',
               'remote',
               'keyboard',
               'cell phone',
               'microwave',
               'oven',
               'toaster',
               'sink',
               'refrigerator',
               'book',
               'clock',
               'vase',
               'scissors',
               'teddy bear',
               'hair drier',
               'toothbrush'
               ]


'''
Got with the following code: 
import h5py

# Load the NYU Depth Dataset .mat file
mat_file = "./resources/nyu_depth_v2_labeled.mat"
with h5py.File(mat_file, "r") as f:
    # Dereference each object in `names` and decode with UTF-16
    names = [f[ref][()].tobytes().decode("utf-16").strip() for ref in f["names"][0]]
    print(f"Class Names: {names}")
'''
NYU_LABELS = ['book', 'bottle', 'cabinet', 'ceiling', 'chair', 'cone', 'counter', 'dishwasher', 'faucet', 'fire extinguisher', 'floor', 'garbage bin', 'microwave', 'paper towel dispenser', 'paper', 'pot', 'refridgerator', 'stove burner', 'table', 'unknown', 'wall', 'bowl', 'magnet', 'sink', 'air vent', 'box', 'door knob', 'door', 'scissor', 'tape dispenser', 'telephone cord', 'telephone', 'track light', 'cork board', 'cup', 'desk', 'laptop', 'air duct', 'basket', 'camera', 'pipe', 'shelves', 'stacked chairs', 'styrofoam object', 'whiteboard', 'computer', 'keyboard', 'ladder', 'monitor', 'stand', 'bar', 'motion camera', 'projector screen', 'speaker', 'bag', 'clock', 'green screen', 'mantel', 'window', 'ball', 'hole puncher', 'light', 'manilla envelope', 'picture', 'mail shelf', 'printer', 'stapler', 'fax machine', 'folder', 'jar', 'magazine', 'ruler', 'cable modem', 'fan', 'file', 'hand sanitizer', 'paper rack', 'vase', 'air conditioner', 'blinds', 'flower', 'plant', 'sofa', 'stereo', 'books', 'exit sign', 'room divider', 'bookshelf', 'curtain', 'projector', 'modem', 'wire', 'water purifier', 'column', 'hooks', 'hanging hooks', 'pen', 'electrical outlet', 'doll', 'eraser', 'pencil holder', 'water carboy', 'mouse', 'cable rack', 'wire rack', 'flipboard', 'map', 'paper cutter', 'tape', 'thermostat', 'heater', 'circuit breaker box', 'paper towel', 'stamp', 'duster', 'poster case', 'whiteboard marker', 'ethernet jack', 'pillow', 'hair brush', 'makeup brush', 'mirror', 'shower curtain', 'toilet', 'toiletries bag', 'toothbrush holder', 'toothbrush', 'toothpaste', 'platter', 'rug', 'squeeze tube', 'shower cap', 'soap', 'towel rod', 'towel', 'bathtub', 'candle', 'tissue box', 'toilet paper', 'container', 'clothes', 'electric toothbrush', 'floor mat', 'lamp', 'drum', 'flower pot', 'banana', 'candlestick', 'shoe', 'stool', 'urn', 'earplugs', 'mailshelf', 'placemat', 'excercise ball', 'alarm clock', 'bed', 'night stand', 'deoderant', 'headphones', 'headboard', 'basketball hoop', 'foot rest', 'laundry basket', 'sock', 'football', 'mens suit', 'cable box', 'dresser', 'dvd player', 'shaver', 'television', 'contact lens solution bottle', 'drawer', 'remote control', 'cologne', 'stuffed animal', 'lint roller', 'tray', 'lock', 'purse', 'toy bottle', 'crate', 'vasoline', 'gift wrapping roll', 'wall decoration', 'hookah', 'radio', 'bicycle', 'pen box', 'mask', 'shorts', 'hat', 'hockey glove', 'hockey stick', 'vuvuzela', 'dvd', 'chessboard', 'suitcase', 'calculator', 'flashcard', 'staple remover', 'umbrella', 'bench', 'yoga mat', 'backpack', 'cd', 'sign', 'hangers', 'notebook', 'hanger', 'security camera', 'folders', 'clothing hanger', 'stairs', 'glass rack', 'saucer', 'tag', 'dolly', 'machine', 'trolly', 'shopping baskets', 'gate', 'bookrack', 'blackboard', 'coffee bag', 'coffee packet', 'hot water heater', 'muffins', 'napkin dispenser', 'plaque', 'plastic tub', 'plate', 'coffee machine', 'napkin holder', 'radiator', 'coffee grinder', 'oven', 'plant pot', 'scarf', 'spice rack', 'stove', 'tea kettle', 'napkin', 'bag of chips', 'bread', 'cutting board', 'dish brush', 'serving spoon', 'sponge', 'toaster', 'cooking pan', 'kitchen items', 'ladel', 'spatula', 'spice stand', 'trivet', 'knife rack', 'knife', 'baking dish', 'dish scrubber', 'drying rack', 'vessel', 'kichen towel', 'tin foil', 'kitchen utensil', 'utensil', 'blender', 'garbage bag', 'sink protector', 'box of ziplock bags', 'spice bottle', 'pitcher', 'pizza box', 'toaster oven', 'step stool', 'vegetable peeler', 'washing machine', 'can opener', 'can of food', 'paper towel holder', 'spoon stand', 'spoon', 'wooden kitchen utensils', 'bag of flour', 'fruit', 'sheet of metal', 'waffle maker', 'cake', 'cell phone', 'tv stand', 'tablecloth', 'wine glass', 'sculpture', 'wall stand', 'iphone', 'coke bottle', 'piano', 'wine rack', 'guitar', 'light switch', 'shirts in hanger', 'router', 'glass pot', 'cart', 'vacuum cleaner', 'bin', 'coins', 'hand sculpture', 'ipod', 'jersey', 'blanket', 'ironing board', 'pen stand', 'mens tie', 'glass baking dish', 'utensils', 'frying pan', 'shopping cart', 'plastic bowl', 'wooden container', 'onion', 'potato', 'jacket', 'dvds', 'surge protector', 'tumbler', 'broom', 'can', 'crock pot', 'person', 'salt shaker', 'wine bottle', 'apple', 'eye glasses', 'menorah', 'bicycle helmet', 'fire alarm', 'water fountain', 'humidifier', 'necklace', 'chandelier', 'barrel', 'chest', 'decanter', 'wooden utensils', 'globe', 'sheets', 'fork', 'napkin ring', 'gift wrapping', 'bed sheets', 'spot light', 'lighting track', 'cannister', 'coffee table', 'mortar and pestle', 'stack of plates', 'ottoman', 'server', 'salt container', 'utensil container', 'phone jack', 'switchbox', 'casserole dish', 'oven handle', 'whisk', 'dish cover', 'electric mixer', 'decorative platter', 'drawer handle', 'fireplace', 'stroller', 'bookend', 'table runner', 'typewriter', 'ashtray', 'key', 'suit jacket', 'range hood', 'cleaning wipes', 'six pack of beer', 'decorative plate', 'watch', 'balloon', 'ipad', 'coaster', 'whiteboard eraser', 'toy', 'toys basket', 'toy truck', 'classroom board', 'chart stand', 'picture of fish', 'plastic box', 'pencil', 'carton', 'walkie talkie', 'binder', 'coat hanger', 'filing shelves', 'plastic crate', 'plastic rack', 'plastic tray', 'flag', 'poster board', 'lunch bag', 'board', 'leg of a girl', 'file holder', 'chart', 'glass pane', 'cardboard tube', 'bassinet', 'toy car', 'toy shelf', 'toy bin', 'toys shelf', 'educational display', 'placard', 'soft toy group', 'soft toy', 'toy cube', 'toy cylinder', 'toy rectangle', 'toy triangle', 'bucket', 'chalkboard', 'game table', 'storage shelvesbooks', 'toy cuboid', 'toy tree', 'wooden toy', 'toy box', 'toy phone', 'toy sink', 'toyhouse', 'notecards', 'toy trucks', 'wall hand sanitizer dispenser', 'cap stand', 'music stereo', 'toys rack', 'display board', 'lid of jar', 'stacked bins  boxes', 'stacked plastic racks', 'storage rack', 'roll of paper towels', 'cables', 'power surge', 'cardboard sheet', 'banister', 'show piece', 'pepper shaker', 'kitchen island', 'excercise equipment', 'treadmill', 'ornamental plant', 'piano bench', 'sheet music', 'grandfather clock', 'iron grill', 'pen holder', 'toy doll', 'globe stand', 'telescope', 'magazine holder', 'file container', 'paper holder', 'flower box', 'pyramid', 'desk mat', 'cordless phone', 'desk drawer', 'envelope', 'window frame', 'id card', 'file stand', 'paper weight', 'toy plane', 'money', 'papers', 'comforter', 'crib', 'doll house', 'toy chair', 'toy sofa', 'plastic chair', 'toy house', 'child carrier', 'cloth bag', 'cradle', 'baby chair', 'chart roll', 'toys box', 'railing', 'clothing dryer', 'clothing washer', 'laundry detergent jug', 'clothing detergent', 'bottle of soap', 'box of paper', 'trolley', 'hand sanitizer dispenser', 'soap holder', 'water dispenser', 'photo', 'water cooler', 'foosball table', 'crayon', 'hoola hoop', 'horse toy', 'plastic toy container', 'pool table', 'game system', 'pool sticks', 'console system', 'video game', 'pool ball', 'trampoline', 'tricycle', 'wii', 'furniture', 'alarm', 'toy table', 'ornamental item', 'copper vessel', 'stick', 'car', 'mezuza', 'toy cash register', 'lid', 'paper bundle', 'business cards', 'clipboard', 'flatbed scanner', 'paper tray', 'mouse pad', 'display case', 'tree sculpture', 'basketball', 'fiberglass case', 'framed certificate', 'cordless telephone', 'shofar', 'trophy', 'cleaner', 'cloth drying stand', 'electric box', 'furnace', 'piece of wood', 'wooden pillar', 'drying stand', 'cane', 'clothing drying rack', 'iron box', 'excercise machine', 'sheet', 'rope', 'sticks', 'wooden planks', 'toilet plunger', 'bar of soap', 'toilet bowl brush', 'light bulb', 'drain', 'faucet handle', 'nailclipper', 'shaving cream', 'rolled carpet', 'clothing iron', 'window cover', 'charger and wire', 'quilt', 'mattress', 'hair dryer', 'stones', 'pepper grinder', 'cat cage', 'dish rack', 'curtain rod', 'calendar', 'head phones', 'cd disc', 'head phone', 'usb drive', 'water heater', 'pan', 'tuna cans', 'baby gate', 'spoon sets', 'cans of cat food', 'cat', 'flower basket', 'fruit platter', 'grapefruit', 'kiwi', 'hand blender', 'knobs', 'vessels', 'cell phone charger', 'wire basket', 'tub of tupperware', 'candelabra', 'litter box', 'shovel', 'cat bed', 'door way', 'belt', 'surge protect', 'glass', 'console controller', 'shoe rack', 'door frame', 'computer disk', 'briefcase', 'mail tray', 'file pad', 'letter stand', 'plastic cup of coffee', 'glass box', 'ping pong ball', 'ping pong racket', 'ping pong table', 'tennis racket', 'ping pong racquet', 'xbox', 'electric toothbrush base', 'toilet brush', 'toiletries', 'razor', 'bottle of contact lens solution', 'contact lens case', 'cream', 'glass container', 'container of skin cream', 'soap dish', 'scale', 'soap stand', 'cactus', 'door  window  reflection', 'ceramic frog', 'incense candle', 'storage space', 'door lock', 'toilet paper holder', 'tissue', 'personal care liquid', 'shower head', 'shower knob', 'knob', 'cream tube', 'perfume box', 'perfume', 'back scrubber', 'door facing trimreflection', 'doorreflection', 'light switchreflection', 'medicine tube', 'wallet', 'soap tray', 'door curtain', 'shower pipe', 'face wash cream', 'flashlight', 'shower base', 'window shelf', 'shower hose', 'toothpaste holder', 'soap box', 'incense holder', 'conch shell', 'roll of toilet paper', 'shower tube', 'bottle of listerine', 'bottle of hand wash liquid', 'tea pot', 'lazy susan', 'avocado', 'fruit stand', 'fruitplate', 'oil container', 'package of water', 'bottle of liquid', 'door way arch', 'jug', 'bulb', 'bagel', 'bag of bagels', 'banana peel', 'bag of oreo', 'flask', 'collander', 'brick', 'torch', 'dog bowl', 'wooden plank', 'eggs', 'grill', 'dog', 'chimney', 'dog cage', 'orange plastic cap', 'glass set', 'vessel set', 'mellon', 'aluminium foil', 'orange', 'peach', 'tea coaster', 'butterfly sculpture', 'corkscrew', 'heating tray', 'food processor', 'corn', 'squash', 'watermellon', 'vegetables', 'celery', 'glass dish', 'hot dogs', 'plastic dish', 'vegetable', 'sticker', 'chapstick', 'sifter', 'fruit basket', 'glove', 'measuring cup', 'water filter', 'wine accessory', 'dishes', 'file box', 'ornamental pot', 'dog toy', 'salt and pepper', 'electrical kettle', 'kitchen container plastic', 'pineapple', 'suger jar', 'steamer', 'charger', 'mug holder', 'orange juicer', 'juicer', 'bag of hot dog buns', 'hamburger bun', 'mug hanger', 'bottle of ketchup', 'toy kitchen', 'food wrapped on a tray', 'kitchen utensils', 'oven mitt', 'bottle of comet', 'wooden utensil', 'decorative dish', 'handle', 'label', 'flask set', 'cooking pot cover', 'tupperware', 'garlic', 'tissue roll', 'lemon', 'wine', 'decorative bottle', 'wire tray', 'tea cannister', 'clothing hamper', 'guitar case', 'wardrobe', 'boomerang', 'button', 'karate belts', 'medal', 'window seat', 'window box', 'necklace holder', 'beeper', 'webcam', 'fish tank', 'luggage', 'life jacket', 'shoelace', 'pen cup', 'eyeball plastic ball', 'toy pyramid', 'model boat', 'certificate', 'puppy toy', 'wire board', 'quill', 'canister', 'toy boat', 'antenna', 'bean bag', 'lint comb', 'travel bag', 'wall divider', 'toy chest', 'headband', 'luggage rack', 'bunk bed', 'lego', 'yarmulka', 'package of bedroom sheets', 'bedding package', 'comb', 'dollar bill', 'pig', 'storage bin', 'storage chest', 'slide', 'playpen', 'electronic drumset', 'ipod dock', 'microphone', 'music keyboard', 'music stand', 'microphone stand', 'album', 'kinect', 'inkwell', 'baseball', 'decorative bowl', 'book holder', 'toy horse', 'desser', 'toy apple', 'toy dog', 'scenary', 'drawer knob', 'shoe hanger', 'tent', 'figurine', 'soccer ball', 'hand weight', 'magic 8ball', 'bottle of perfume', 'sleeping bag', 'decoration item', 'envelopes', 'trinket', 'hand fan', 'sculpture of the chrysler building', 'sculpture of the eiffel tower', 'sculpture of the empire state building', 'jeans', 'garage door', 'case', 'rags', 'decorative item', 'toy stroller', 'shelf frame', 'cat house', 'can of beer', 'dog bed', 'lamp shade', 'bracelet', 'reflection of window shutters', 'decorative egg', 'indoor fountain', 'photo album', 'decorative candle', 'walkietalkie', 'serving dish', 'floor trim', 'mini display platform', 'american flag', 'vhs tapes', 'throw', 'newspapers', 'mantle', 'package of bottled water', 'serving platter', 'display platter', 'centerpiece', 'tea box', 'gold piece', 'wreathe', 'lectern', 'hammer', 'matchbox', 'pepper', 'yellow pepper', 'duck', 'eggplant', 'glass ware', 'sewing machine', 'rolled up rug', 'doily', 'coffee pot', 'torah']

NUM_CLASSES = 20 if DATASET != 'NYUv2' else NYU_LABELS.__len__()

C:\Users\19578\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## utils.py

In [2]:
# utils.py

import os
import random
from collections import Counter

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm


def iou_width_height(boxes1, boxes2):
    """
    Parameters:
        boxes1 (tensor): width and height of the first bounding boxes
        boxes2 (tensor): width and height of the second bounding boxes
    Returns:
        tensor: Intersection over union of the corresponding boxes
    """
    intersection = torch.min(boxes1[..., 0], boxes2[..., 0]) * torch.min(
        boxes1[..., 1], boxes2[..., 1]
    )
    union = (
        boxes1[..., 0] * boxes1[..., 1] +
        boxes2[..., 0] * boxes2[..., 1] - intersection
    )
    return intersection / union


def intersection_over_union(boxes_preds, boxes_labels, box_format="midpoint"):
    """
    Video explanation of this function:
    https://youtu.be/XXYG5ZWtjj0

    This function calculates intersection over union (iou) given pred boxes
    and target boxes.

    Parameters:
        boxes_preds (tensor): Predictions of Bounding Boxes (BATCH_SIZE, 4)
        boxes_labels (tensor): Correct labels of Bounding Boxes (BATCH_SIZE, 4)
        box_format (str): midpoint/corners, if boxes (x,y,w,h) or (x1,y1,x2,y2)

    Returns:
        tensor: Intersection over union for all examples
    """

    if box_format == "midpoint":
        box1_x1 = boxes_preds[..., 0:1] - boxes_preds[..., 2:3] / 2
        box1_y1 = boxes_preds[..., 1:2] - boxes_preds[..., 3:4] / 2
        box1_x2 = boxes_preds[..., 0:1] + boxes_preds[..., 2:3] / 2
        box1_y2 = boxes_preds[..., 1:2] + boxes_preds[..., 3:4] / 2
        box2_x1 = boxes_labels[..., 0:1] - boxes_labels[..., 2:3] / 2
        box2_y1 = boxes_labels[..., 1:2] - boxes_labels[..., 3:4] / 2
        box2_x2 = boxes_labels[..., 0:1] + boxes_labels[..., 2:3] / 2
        box2_y2 = boxes_labels[..., 1:2] + boxes_labels[..., 3:4] / 2

    if box_format == "corners":
        box1_x1 = boxes_preds[..., 0:1]
        box1_y1 = boxes_preds[..., 1:2]
        box1_x2 = boxes_preds[..., 2:3]
        box1_y2 = boxes_preds[..., 3:4]
        box2_x1 = boxes_labels[..., 0:1]
        box2_y1 = boxes_labels[..., 1:2]
        box2_x2 = boxes_labels[..., 2:3]
        box2_y2 = boxes_labels[..., 3:4]

    x1 = torch.max(box1_x1, box2_x1)
    y1 = torch.max(box1_y1, box2_y1)
    x2 = torch.min(box1_x2, box2_x2)
    y2 = torch.min(box1_y2, box2_y2)

    intersection = (x2 - x1).clamp(0) * (y2 - y1).clamp(0)
    box1_area = abs((box1_x2 - box1_x1) * (box1_y2 - box1_y1))
    box2_area = abs((box2_x2 - box2_x1) * (box2_y2 - box2_y1))

    return intersection / (box1_area + box2_area - intersection + 1e-6)


def non_max_suppression(bboxes, iou_threshold, threshold, box_format="corners"):
    """
    Video explanation of this function:
    https://youtu.be/YDkjWEN8jNA

    Does Non Max Suppression given bboxes

    Parameters:
        bboxes (list): list of lists containing all bboxes with each bboxes
        specified as [class_pred, prob_score, x1, y1, x2, y2]
        iou_threshold (float): threshold where predicted bboxes is correct
        threshold (float): threshold to remove predicted bboxes (independent of IoU)
        box_format (str): "midpoint" or "corners" used to specify bboxes

    Returns:
        list: bboxes after performing NMS given a specific IoU threshold
    """

    assert type(bboxes) == list

    bboxes = [box for box in bboxes if box[1] > threshold]
    bboxes = sorted(bboxes, key=lambda x: x[1], reverse=True)
    bboxes_after_nms = []

    while bboxes:
        chosen_box = bboxes.pop(0)

        bboxes = [
            box
            for box in bboxes
            if box[0] != chosen_box[0]
            or intersection_over_union(
                torch.tensor(chosen_box[2:]),
                torch.tensor(box[2:]),
                box_format=box_format,
            )
            < iou_threshold
        ]

        bboxes_after_nms.append(chosen_box)

    return bboxes_after_nms


def mean_average_precision(
        pred_boxes, true_boxes, iou_threshold=0.5, box_format="midpoint", num_classes=20
):
    """
    Video explanation of this function:
    https://youtu.be/FppOzcDvaDI

    This function calculates mean average precision (mAP)

    Parameters:
        pred_boxes (list): list of lists containing all bboxes with each bboxes
        specified as [train_idx, class_prediction, prob_score, x1, y1, x2, y2]
        true_boxes (list): Similar as pred_boxes except all the correct ones
        iou_threshold (float): threshold where predicted bboxes is correct
        box_format (str): "midpoint" or "corners" used to specify bboxes
        num_classes (int): number of classes

    Returns:
        float: mAP value across all classes given a specific IoU threshold
    """

    # list storing all AP for respective classes
    average_precisions = []

    # used for numerical stability later on
    epsilon = 1e-6

    for c in range(num_classes):
        detections = []
        ground_truths = []

        # Go through all predictions and targets,
        # and only add the ones that belong to the
        # current class c
        for detection in pred_boxes:
            if detection[1] == c:
                detections.append(detection)

        for true_box in true_boxes:
            if true_box[1] == c:
                ground_truths.append(true_box)

        # find the amount of bboxes for each training example
        # Counter here finds how many ground truth bboxes we get
        # for each training example, so let's say img 0 has 3,
        # img 1 has 5 then we will obtain a dictionary with:
        # amount_bboxes = {0:3, 1:5}
        amount_bboxes = Counter([gt[0] for gt in ground_truths])

        # We then go through each key, val in this dictionary
        # and convert to the following (w.r.t same example):
        # ammount_bboxes = {0:torch.tensor[0,0,0], 1:torch.tensor[0,0,0,0,0]}
        for key, val in amount_bboxes.items():
            amount_bboxes[key] = torch.zeros(val)

        # sort by box probabilities which is index 2
        detections.sort(key=lambda x: x[2], reverse=True)
        TP = torch.zeros((len(detections)))
        FP = torch.zeros((len(detections)))
        total_true_bboxes = len(ground_truths)

        # If none exists for this class then we can safely skip
        if total_true_bboxes == 0:
            continue

        for detection_idx, detection in enumerate(detections):
            # Only take out the ground_truths that have the same
            # training idx as detection
            ground_truth_img = [
                bbox for bbox in ground_truths if bbox[0] == detection[0]
            ]

            num_gts = len(ground_truth_img)
            best_iou = 0

            for idx, gt in enumerate(ground_truth_img):
                iou = intersection_over_union(
                    torch.tensor(detection[3:]),
                    torch.tensor(gt[3:]),
                    box_format=box_format,
                )

                if iou > best_iou:
                    best_iou = iou
                    best_gt_idx = idx

            if best_iou > iou_threshold:
                # only detect ground truth detection once
                if amount_bboxes[detection[0]][best_gt_idx] == 0:
                    # true positive and add this bounding box to seen
                    TP[detection_idx] = 1
                    amount_bboxes[detection[0]][best_gt_idx] = 1
                else:
                    FP[detection_idx] = 1

            # if IOU is lower then the detection is a false positive
            else:
                FP[detection_idx] = 1

        TP_cumsum = torch.cumsum(TP, dim=0)
        FP_cumsum = torch.cumsum(FP, dim=0)
        recalls = TP_cumsum / (total_true_bboxes + epsilon)
        precisions = TP_cumsum / (TP_cumsum + FP_cumsum + epsilon)
        precisions = torch.cat((torch.tensor([1]), precisions))
        recalls = torch.cat((torch.tensor([0]), recalls))
        # torch.trapz for numerical integration
        average_precisions.append(torch.trapz(precisions, recalls))

    return sum(average_precisions) / len(average_precisions)


def plot_image(image, boxes):
    """Plots predicted bounding boxes on the image"""
    cmap = plt.get_cmap("tab20b")
    if DATASET == 'COCO':
        class_labels = COCO_LABELS
    elif DATASET == 'NYUv2':
        class_labels = NYU_LABELS
    else:
        class_labels = PASCAL_CLASSES
    colors = [cmap(i) for i in np.linspace(0, 1, len(class_labels))]
    im = np.array(image)
    height, width, _ = im.shape

    # Create figure and axes
    fig, ax = plt.subplots(1)
    # Display the image
    ax.imshow(im)

    # box[0] is x midpoint, box[2] is width
    # box[1] is y midpoint, box[3] is height

    # Create a Rectangle patch
    for box in boxes:
        assert len(
            box) == 6, "box should contain class pred, confidence, x, y, width, height"
        class_pred = box[0]
        box = box[2:]
        upper_left_x = box[0] - box[2] / 2
        upper_left_y = box[1] - box[3] / 2
        rect = patches.Rectangle(
            (upper_left_x * width, upper_left_y * height),
            box[2] * width,
            box[3] * height,
            linewidth=2,
            edgecolor=colors[int(class_pred)],
            facecolor="none",
        )
        # Add the patch to the Axes
        ax.add_patch(rect)
        plt.text(
            upper_left_x * width,
            upper_left_y * height,
            s=class_labels[int(class_pred)],
            color="white",
            verticalalignment="top",
            bbox={"color": colors[int(class_pred)], "pad": 0},
        )

    plt.show()


def get_evaluation_bboxes(
        loader,
        model,
        iou_threshold,
        anchors,
        threshold,
        box_format="midpoint",
        device="cuda",
):
    # make sure model is in eval before get bboxes
    model.eval()
    train_idx = 0
    all_pred_boxes = []
    all_true_boxes = []
    for batch_idx, (x, labels) in enumerate(tqdm(loader)):
        x = x.to(device)

        with torch.no_grad():
            predictions = model(x)

        batch_size = x.shape[0]
        bboxes = [[] for _ in range(batch_size)]
        for i in range(3):
            S = predictions[i].shape[2]
            anchor = torch.tensor([*anchors[i]]).to(device) * S
            boxes_scale_i = cells_to_bboxes(
                predictions[i], anchor, S=S, is_preds=True
            )
            for idx, (box) in enumerate(boxes_scale_i):
                bboxes[idx] += box

        # we just want one bbox for each label, not one for each scale
        true_bboxes = cells_to_bboxes(
            labels[2], anchor, S=S, is_preds=False
        )

        for idx in range(batch_size):
            nms_boxes = non_max_suppression(
                bboxes[idx],
                iou_threshold=iou_threshold,
                threshold=threshold,
                box_format=box_format,
            )

            for nms_box in nms_boxes:
                all_pred_boxes.append([train_idx] + nms_box)

            for box in true_bboxes[idx]:
                if box[1] > threshold:
                    all_true_boxes.append([train_idx] + box)

            train_idx += 1

    model.train()
    return all_pred_boxes, all_true_boxes


def cells_to_bboxes(predictions, anchors, S, is_preds=True):
    """
    Scales the predictions coming from the model to
    be relative to the entire image such that they for example later
    can be plotted or.
    INPUT:
    predictions: tensor of size (N, 3, S, S, num_classes+5)
    anchors: the anchors used for the predictions
    S: the number of cells the image is divided in on the width (and height)
    is_preds: whether the input is predictions or the true bounding boxes
    OUTPUT:
    converted_bboxes: the converted boxes of sizes (N, num_anchors, S, S, 1+5) with class index,
                      object score, bounding box coordinates
    """
    BATCH_SIZE = predictions.shape[0]
    num_anchors = len(anchors)
    box_predictions = predictions[..., 1:5]
    if is_preds:
        anchors = anchors.reshape(1, len(anchors), 1, 1, 2)
        box_predictions[..., 0:2] = torch.sigmoid(box_predictions[..., 0:2])
        box_predictions[..., 2:] = torch.exp(
            box_predictions[..., 2:]) * anchors
        scores = torch.sigmoid(predictions[..., 0:1])
        best_class = torch.argmax(predictions[..., 5:], dim=-1).unsqueeze(-1)
    else:
        scores = predictions[..., 0:1]
        best_class = predictions[..., 5:6]

    cell_indices = (
        torch.arange(S)
        .repeat(predictions.shape[0], 3, S, 1)
        .unsqueeze(-1)
        .to(predictions.device)
    )
    x = 1 / S * (box_predictions[..., 0:1] + cell_indices)
    y = 1 / S * (box_predictions[..., 1:2] +
                 cell_indices.permute(0, 1, 3, 2, 4))
    w_h = 1 / S * box_predictions[..., 2:4]
    converted_bboxes = torch.cat(
        (best_class, scores, x, y, w_h), dim=-1).reshape(BATCH_SIZE, num_anchors * S * S, 6)
    return converted_bboxes.tolist()


def check_class_accuracy(model, loader, threshold):
    model.eval()
    tot_class_preds, correct_class = 0, 0
    tot_noobj, correct_noobj = 0, 0
    tot_obj, correct_obj = 0, 0

    for idx, (x, y) in enumerate(tqdm(loader)):
        x = x.to(DEVICE)
        with torch.no_grad():
            out = model(x)

        for i in range(3):
            y[i] = y[i].to(DEVICE)
            obj = y[i][..., 0] == 1  # in paper this is Iobj_i
            noobj = y[i][..., 0] == 0  # in paper this is Iobj_i

            correct_class += torch.sum(
                torch.argmax(out[i][..., 5:][obj], dim=-1) == y[i][..., 5][obj]
            )
            tot_class_preds += torch.sum(obj)

            obj_preds = torch.sigmoid(out[i][..., 0]) > threshold
            correct_obj += torch.sum(obj_preds[obj] == y[i][..., 0][obj])
            tot_obj += torch.sum(obj)
            correct_noobj += torch.sum(obj_preds[noobj] == y[i][..., 0][noobj])
            tot_noobj += torch.sum(noobj)

    print(
        f"Class accuracy is: {(correct_class / (tot_class_preds + 1e-16)) * 100:2f}%")
    print(
        f"No obj accuracy is: {(correct_noobj / (tot_noobj + 1e-16)) * 100:2f}%")
    print(f"Obj accuracy is: {(correct_obj / (tot_obj + 1e-16)) * 100:2f}%")
    model.train()


def get_mean_std(loader):
    # var[X] = E[X**2] - E[X]**2
    channels_sum, channels_sqrd_sum, num_batches = 0, 0, 0

    for data, _ in tqdm(loader):
        channels_sum += torch.mean(data, dim=[0, 2, 3])
        channels_sqrd_sum += torch.mean(data ** 2, dim=[0, 2, 3])
        num_batches += 1

    mean = channels_sum / num_batches
    std = (channels_sqrd_sum / num_batches - mean ** 2) ** 0.5

    return mean, std


def save_checkpoint(model, optimizer, filename="my_checkpoint.pth.tar"):
    print("=> Saving checkpoint")
    checkpoint = {
        "state_dict": model.state_dict(),
        "optimizer": optimizer.state_dict(),
    }
    torch.save(checkpoint, filename)


def load_checkpoint(checkpoint_file, model, optimizer, lr):
    print("=> Loading checkpoint")
    checkpoint = torch.load(checkpoint_file, map_location=DEVICE)
    model.load_state_dict(checkpoint["state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer"])

    # If we don't do this then it will just have learning rate of old checkpoint
    # and it will lead to many hours of debugging \:
    for param_group in optimizer.param_groups:
        param_group["lr"] = lr

def get_loaders(train_csv_path, test_csv_path):

    train_dataset = YOLODataset(
        train_csv_path,
        transform=train_transforms,
        S=[IMAGE_SIZE // 32, IMAGE_SIZE // 16, IMAGE_SIZE // 8],
        img_dir=IMG_DIR,
        label_dir=LABEL_DIR,
        anchors=ANCHORS,
    )
    test_dataset = YOLODataset(
        test_csv_path,
        transform=test_transforms,
        S=[IMAGE_SIZE // 32, IMAGE_SIZE // 16, IMAGE_SIZE // 8],
        img_dir=IMG_DIR,
        label_dir=LABEL_DIR,
        anchors=ANCHORS,
    )
    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        shuffle=True,
        drop_last=False,
    )
    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        shuffle=False,
        drop_last=False,
    )

    train_eval_dataset = YOLODataset(
        train_csv_path,
        transform=test_transforms,
        S=[IMAGE_SIZE // 32, IMAGE_SIZE // 16, IMAGE_SIZE // 8],
        img_dir=IMG_DIR,
        label_dir=LABEL_DIR,
        anchors=ANCHORS,
    )
    train_eval_loader = DataLoader(
        dataset=train_eval_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        shuffle=False,
        drop_last=False,
    )

    return train_loader, test_loader, train_eval_loader

def get_loaders_nyu(mat_file_path, train_ratio=0.8):
    """
    Creates data loaders for the NYU Depth Dataset with a train-test split.

    Args:
        mat_file_path (str): Path to the .mat file containing the NYU dataset.
        train_ratio (float): Proportion of the dataset to use for training.

    Returns:
        train_loader (DataLoader): DataLoader for training data.
        test_loader (DataLoader): DataLoader for testing data.
        train_eval_loader (DataLoader): DataLoader for evaluating training data.
    """
    # Load the number of samples in the dataset
    import h5py
    with h5py.File(mat_file_path, "r") as f:
        num_samples = len(f["images"])

    # Generate train and test indices
    train_indices, test_indices = generate_train_test_indices(num_samples, train_ratio)

    # Create training and testing datasets
    train_dataset = NYUYoloDataset(
        mat_file=mat_file_path,
        anchors=ANCHORS,
        image_size=IMAGE_SIZE,
        S=[IMAGE_SIZE // 32, IMAGE_SIZE // 16, IMAGE_SIZE // 8],
        C=40,
        transform=train_transforms,
        indices=train_indices,
    )
    test_dataset = NYUYoloDataset(
        mat_file=mat_file_path,
        anchors=ANCHORS,
        image_size=IMAGE_SIZE,
        S=[IMAGE_SIZE // 32, IMAGE_SIZE // 16, IMAGE_SIZE // 8],
        C=40,
        transform=test_transforms,
        indices=test_indices,
    )
    train_eval_dataset = NYUYoloDataset(
        mat_file=mat_file_path,
        anchors=ANCHORS,
        image_size=IMAGE_SIZE,
        S=[IMAGE_SIZE // 32, IMAGE_SIZE // 16, IMAGE_SIZE // 8],
        C=40,
        transform=test_transforms,
        indices=train_indices,
    )

    # Create DataLoaders
    train_loader = DataLoader(
        dataset=train_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        shuffle=True,
        drop_last=False,
    )
    test_loader = DataLoader(
        dataset=test_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        shuffle=False,
        drop_last=False,
    )
    train_eval_loader = DataLoader(
        dataset=train_eval_dataset,
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        shuffle=False,
        drop_last=False,
    )

    return train_loader, test_loader, train_eval_loader


def plot_couple_examples(model, loader, thresh, iou_thresh, anchors):
    model.eval()
    x, y = next(iter(loader))
    x = x.to("cuda")
    with torch.no_grad():
        out = model(x)
        bboxes = [[] for _ in range(x.shape[0])]
        for i in range(3):
            batch_size, A, S, _, _ = out[i].shape
            anchor = anchors[i]
            boxes_scale_i = cells_to_bboxes(
                out[i], anchor, S=S, is_preds=True
            )
            for idx, (box) in enumerate(boxes_scale_i):
                bboxes[idx] += box

        model.train()

    for i in range(batch_size):
        nms_boxes = non_max_suppression(
            bboxes[i], iou_threshold=iou_thresh, threshold=thresh, box_format="midpoint",
        )
        plot_image(x[i].permute(1, 2, 0).detach().cpu(), nms_boxes)

def generate_train_test_indices(num_samples, train_ratio=0.8):
    """
    Splits indices into training and testing subsets.

    Args:
        num_samples (int): Total number of samples in the dataset.
        train_ratio (float): Proportion of samples to use for training.

    Returns:
        train_indices (numpy.ndarray): Indices for the training subset.
        test_indices (numpy.ndarray): Indices for the testing subset.
    """
    indices = np.arange(num_samples)
    np.random.shuffle(indices)

    split_idx = int(num_samples * train_ratio)
    train_indices = indices[:split_idx]
    test_indices = indices[split_idx:]

    return train_indices, test_indices

def seed_everything(seed=42):
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## model.py

In [3]:
# model.py

import torch
import torch.nn as nn

"""
Tuple: (filters, kernel_size, stride) 
List: ["B", number of repeats]a
"U": upsampling the feature map and concatenating with a previous layer
"S": scale prediction block and computing the YOLO loss
Note: Every convolution is a "same" convolution 
"""
config = [
    (32, 3, 1),
    (64, 3, 2),
    ["B", 1],
    (128, 3, 2),
    ["B", 2],
    (256, 3, 2),
    ["B", 8],
    # first route from the end of the previous block
    (512, 3, 2),
    ["B", 8],
    # second route from the end of the previous block
    (1024, 3, 2),
    ["B", 4],
    # To this point is Darknet-53
    (512, 1, 1),
    (1024, 3, 1),
    "S",
    (256, 1, 1),
    "U",
    (256, 1, 1),
    (512, 3, 1),
    "S",
    (128, 1, 1),
    "U",
    (128, 1, 1),
    (256, 3, 1),
    "S",
]


class CNNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, bn_act=True, **kwargs):
        super(CNNBlock, self).__init__()
        self.conv = nn.Conv2d(in_channels, out_channels,
                              bias=not bn_act, **kwargs)
        self.bn = nn.BatchNorm2d(out_channels)
        self.leaky = nn.LeakyReLU(0.1)
        self.use_bn_act = bn_act

    def forward(self, x):
        if self.use_bn_act:
            return self.leaky(self.bn(self.conv(x)))
        else:
            return self.conv(x)


class ResidualBlock(nn.Module):
    def __init__(self, channels, use_residual=True, num_repeats=1):
        super(ResidualBlock, self).__init__()
        self.layers = nn.ModuleList()
        for repeat in range(num_repeats):
            self.layers.append(
                nn.Sequential(
                    CNNBlock(channels, channels // 2, kernel_size=1),
                    CNNBlock(channels // 2, channels,
                             kernel_size=3, padding=1)
                )
            )
        self.use_residual = use_residual
        self.num_repeats = num_repeats

    def forward(self, x):
        for layer in self.layers:
            if self.use_residual:
                x = x + layer(x)
            else:
                x = layer(x)
        return x


class ScalePrediction(nn.Module):
    def __init__(self, in_channels, num_classes):
        super(ScalePrediction, self).__init__()
        self.pred = nn.Sequential(
            CNNBlock(in_channels, 2 * in_channels, kernel_size=3, padding=1),
            CNNBlock(2 * in_channels, (num_classes + 5)
                     * 3, bn_act=False, kernel_size=1)
        )
        self.num_classes = num_classes

    def forward(self, x):
        return (
            self.pred(x)
            .reshape(x.shape[0], 3, self.num_classes + 5, x.shape[2], x.shape[3])
            .permute(0, 1, 3, 4, 2)
        )


class YOLOv3(nn.Module):
    def __init__(self, in_channels=3, num_classes=20):
        super(YOLOv3, self).__init__()
        self.num_classes = num_classes
        self.in_channels = in_channels
        self.layers = self._create_conv_layers()

    def forward(self, x):
        outputs = []
        route_connections = []

        for layer in self.layers:
            if isinstance(layer, ScalePrediction):
                outputs.append(layer(x))
                continue

            x = layer(x)

            if isinstance(layer, ResidualBlock) and layer.num_repeats == 8:
                route_connections.append(x)

            elif isinstance(layer, nn.Upsample):
                x = torch.cat([x, route_connections[-1]], dim=1)
                route_connections.pop()
        return outputs

    def _create_conv_layers(self):
        layers = nn.ModuleList()
        in_channels = self.in_channels

        for module in config:
            if isinstance(module, tuple):
                out_channels, kernel_size, stride = module
                layers.append(CNNBlock(in_channels,
                                       out_channels,
                                       kernel_size=kernel_size,
                                       stride=stride,
                                       padding=1 if kernel_size == 3 else 0))
                in_channels = out_channels

            elif isinstance(module, list):
                num_repeats = module[1]
                layers.append(ResidualBlock(
                    in_channels, num_repeats=num_repeats))

            elif isinstance(module, str):
                if module == "S":
                    layers += [
                        ResidualBlock(
                            in_channels, use_residual=False, num_repeats=1),
                        CNNBlock(in_channels, in_channels // 2, kernel_size=1),
                        ScalePrediction(in_channels // 2,
                                        num_classes=self.num_classes)
                    ]
                    in_channels = in_channels // 2

                elif module == "U":
                    layers.append(nn.Upsample(scale_factor=2, mode='bilinear'))
                    in_channels = in_channels * 3

        return layers


def test():
    num_classes = 20
    model = YOLOv3(num_classes=num_classes)
    img_size = 416
    x = torch.randn((2, 3, img_size, img_size))
    out = model(x)
    assert out[0].shape == (2, 3, img_size // 32,
                            img_size // 32, 5 + num_classes)
    assert out[1].shape == (2, 3, img_size // 16,
                            img_size // 16, 5 + num_classes)
    assert out[2].shape == (2, 3, img_size // 8,
                            img_size // 8, 5 + num_classes)
    print("Success!")


if __name__ == '__main__':
    test()

Success!


## dataset.py

In [4]:
# dataset.py

import os

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageFile
from torch.utils.data import Dataset, DataLoader

# from utils import (
#     cells_to_bboxes,
#     iou_width_height,
#     non_max_suppression as nms,
#     plot_image
# )

ImageFile.LOAD_TRUNCATED_IMAGES = True


class YOLODataset(Dataset):
    def __init__(self, csv_file, img_dir, label_dir, anchors, image_size=416, S=None,
                 C=20, transform=None):
        '''
        csv_file: path to the csv file containing image paths and annotations
        img_dir: directory containing the images
        label_dir: directory containing the labels
        anchors: list containing the anchors
        image_size: size of the image
        S: list containing the sizes of the three scales
        C: number of classes
        transform: data augmentation
        '''
        if S is None:
            S = [13, 26, 52]
        self.annotations = pd.read_csv(csv_file)
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.image_size = image_size
        self.transform = transform
        self.S = S
        self.C = C

        # anchors[i] -> [x, y] for 3 anchors in the ith scale.
        # Hence, anchors[0] + anchors[1] + anchors[2] -> will just put the coordinates of all anchors in one single list
        self.anchors = torch.tensor(anchors[0] + anchors[1] + anchors[2])

        self.num_anchors = self.anchors.shape[0]
        # As there are three different prediction scales.
        self.num_anchors_per_scale = self.num_anchors // 3
        self.ignore_iou_thresh = 0.5

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, index):
        """
        Steps involved:
        1) Load image and corresponding text file for labels.
        2) Create `target` (a regular Python list): 
            - For each scale, we have a 4D array: 
                - 1st dim: anchor box index
                - 2nd dim: scale dim 1
                - 3rd dim: scale dim 2
                - 4th dim: Target values (objectness_score, x, y, width, height, class_label)
        3) For every bounding box for an image: 
            - calculate IoU score for this box with all the anchor boxes. Argsort the anchor indices based on their IoU scores in descending order. 
            - For every anchor box: 
                - find the actual anchor and the scale it belongs to. 
                - calculate the cell coordinates of this box in this scale.
                - See if this anchor in this scale at this cell coordinates has been assigned to some other object before. If yes, skip this box. 
                - Else, mark the objectness score for this target position as 1. 
        """
        img_path = os.path.join(self.img_dir, self.annotations.iloc[index, 0])
        image = np.array(Image.open(img_path).convert("RGB"))

        label_path = os.path.join(
            self.label_dir, self.annotations.iloc[index, 1])
        # Label Format: [class_label, x, y, w, h] | Expected Format: [x, y, w, h, class_label]
        bboxes = np.roll(np.loadtxt(
            label_path, delimiter=" ", ndmin=2), 4, axis=1)

        # data augmentation
        if self.transform:
            augmentations = self.transform(image=image, bboxes=bboxes)
            image = augmentations["image"]
            bboxes = augmentations["bboxes"]

        # assumption: Each scale has 3 anchors and all scales have the same number of anchors.
        # 6 = object score + x + y + anchor height + anchor width + class label
        # For YoLov3, scale = [13, 26, 52]
        targets = [torch.zeros((self.num_anchors // 3, scale, scale, 6))
                   for scale in self.S]
        for box in bboxes:
            '''
            Since the grid cells are quite small, for easier IOU calculation, we can assume that anchor in a cell 
            and the ground truth bounding box's midpoint share the same midpoint. Hence, we can use iou_width_height.
            '''
            iou_anchors = iou_width_height(
                torch.tensor(box[2:4]), self.anchors)
            # So that we can get the index of the anchor box with the largest IOU with the target box.
            anchor_indices = iou_anchors.argsort(descending=True, dim=0)

            x, y, width, height, class_label = box

            # It is also assumed that each bounding box will be associated with only one class label.
            has_anchor = [False] * 3
            for anchor_index in anchor_indices:
                scale_idx = anchor_index // self.num_anchors_per_scale  # which scale
                S = self.S[scale_idx]

                # which anchor in this scale
                anchor_on_scale = anchor_index % self.num_anchors_per_scale

                # finds out the target cell of the image in this scale
                i, j = int(S * y), int(S * x)
                # choose a scale -> choose an anchor on this scale -> choose the cell in this scale
                # -> prob. of existence of an object
                anchor_taken = targets[scale_idx][anchor_on_scale, i, j, 0]

                if not anchor_taken and not has_anchor[scale_idx]:
                    targets[scale_idx][anchor_on_scale, i, j, 0] = 1
                    # relative position of the object's center wrt the assigned grid cell
                    x_cell, y_cell = S * x - j, S * y - i
                    width_cell, height_cell = width * S, height * S
                    box_coordinates = torch.tensor(
                        [x_cell, y_cell, width_cell, height_cell])

                    targets[scale_idx][anchor_on_scale,
                                       i, j, 1:5] = box_coordinates
                    targets[scale_idx][anchor_on_scale,
                                       i, j, 5] = int(class_label)
                    has_anchor[scale_idx] = True

                elif not anchor_taken and iou_anchors[anchor_index] > self.ignore_iou_thresh:
                    targets[scale_idx][anchor_on_scale, i, j, 0] = -1

        return image, tuple(targets)


def test():
    anchors = ANCHORS

    transform = train_transforms

    dataset = YOLODataset(
        DATASET + '/train.csv',
        IMG_DIR,
        LABEL_DIR,
        S=[13, 26, 52],
        anchors=anchors,
        transform=transform,
    )
    S = [13, 26, 52]
    scaled_anchors = torch.tensor(anchors) / (
        1 / torch.tensor(S).unsqueeze(1).unsqueeze(1).repeat(1, 3, 2)
    )
    loader = DataLoader(dataset=dataset, batch_size=1, shuffle=True)
    for x, y in loader:
        boxes = []

        for i in range(y[0].shape[1]):
            anchor = scaled_anchors[i]
            boxes += cells_to_bboxes(
                y[i], is_preds=False, S=y[i].shape[2], anchors=anchor
            )[0]
        boxes = non_max_suppression(boxes, iou_threshold=1,
                    threshold=0.7, box_format="midpoint")
        print(boxes)
        plot_image(x[0].permute(1, 2, 0).to("cpu"), boxes)


# if __name__ == "__main__":
#     test()

## NYU Depth Dataset

In [5]:
import os
import numpy as np
import torch
from torch.utils.data import Dataset
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2

class NYUYoloDataset(Dataset):
    def __init__(self, mat_file, anchors, image_size=416, S=None, C=40, transform=None, indices=None):
        """
        Args:
            mat_file (str): Path to the .mat file containing the dataset.
            anchors (list): List of anchors for each scale.
            image_size (int): Size to which images will be resized.
            S (list): List of scales (e.g., [13, 26, 52] for YOLOv3).
            C (int): Number of classes in the dataset.
            transform (callable, optional): Transformations to apply to the images and labels.
            indices (list, optional): List of indices specifying the subset of the dataset to use.
        """
        import h5py

        with h5py.File(mat_file, "r") as f:
            self.data = {key: np.array(f[key]) for key in f.keys()}
            self.images = np.rot90(self.data["images"], k=-1, axes=(2, 3))  # (N, 3, H, W), Rotate HxW axes
            self.instances = np.rot90(self.data["instances"], k=-1, axes=(1, 2)) # (N, H, W)
            self.labels = np.rot90(self.data["labels"], k=-1, axes=(1, 2)) # (N, H, W)
                # Use only the specified subset of data if indices are provided
        if indices is not None:
            self.images = self.images[indices]
            self.instances = self.instances[indices]
            self.labels = self.labels[indices]
        self.anchors = torch.tensor(anchors[0] + anchors[1] + anchors[2])
        self.num_anchors = self.anchors.shape[0]
        self.num_anchors_per_scale = self.num_anchors // 3
        self.ignore_iou_thresh = 0.5
        self.image_size = image_size
        self.S = S if S else [13, 26, 52]
        self.C = C
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, index):
        """
        Returns:
            image: Tensor of shape [3, img_size, img_size]
            targets: List of tensors for each scale.
        """
        # Load image, instance map, and label map
        image = self.images[index]  # (C, H, W)
        instance_map = self.instances[index]  # (H, W)
        label_map = self.labels[index]  # (H, W)

        # Convert image to (H, W, C) and scale to [0, 255]
        image = np.transpose(image, (1, 2, 0)).astype(np.uint8)

        # Extract bounding boxes
        bboxes = self._get_bounding_boxes(instance_map, label_map)

        # Apply transformations
        if self.transform:
            augmentations = self.transform(image=image, bboxes=bboxes)
            image = augmentations["image"]
            bboxes = augmentations["bboxes"]

        # Prepare YOLO targets
        targets = [torch.zeros((self.num_anchors_per_scale, scale, scale, 6)) for scale in self.S]
        for box in bboxes:
            iou_anchors = iou_width_height(torch.tensor(box[2:4]), self.anchors)
            anchor_indices = iou_anchors.argsort(descending=True)
            x, y, width, height, class_label = box

            has_anchor = [False] * len(self.S)
            for anchor_index in anchor_indices:
                scale_idx = int(anchor_index // self.num_anchors_per_scale)
                S = self.S[scale_idx]
                anchor_on_scale = int(anchor_index % self.num_anchors_per_scale)
                i, j = int(S * y), int(S * x)

                if not has_anchor[scale_idx]:
                    x_cell, y_cell = S * x - j, S * y - i
                    width_cell, height_cell = width * S, height * S
                    box_coordinates = torch.tensor([x_cell, y_cell, width_cell, height_cell])
                    targets[scale_idx][anchor_on_scale, i, j, 0] = 1
                    targets[scale_idx][anchor_on_scale, i, j, 1:5] = box_coordinates
                    targets[scale_idx][anchor_on_scale, i, j, 5] = int(class_label)
                    has_anchor[scale_idx] = True
                elif not has_anchor[scale_idx] and iou_anchors[anchor_index] > self.ignore_iou_thresh:
                    targets[scale_idx][anchor_on_scale, i, j, 0] = -1

        return image, tuple(targets)
    
    def _get_instance_masks(self, img_object_labels, img_instances):
        '''
        Extracts instance masks and labels from the object labels and instance maps.
        Args:
            imgObjectLabels (numpy.ndarray): 2D array (H, W) with object labels for each pixel.
            imgInstances (numpy.ndarray): 2D array (H, W) with instance IDs for each pixel.
        Returns:
            instanceMasks (numpy.ndarray): 3D array (H, W, N) with binary masks for each instance.
            instanceLabels (numpy.ndarray): 1D array (N,) with class labels for each instance.
        '''
        H, W = img_object_labels.shape

        # Find unique (label, instance_id) pairs
        pairs = np.unique(np.stack([img_object_labels.ravel(), img_instances.ravel()], axis=1), axis=0)
        pairs = pairs[np.sum(pairs, axis=1) != 0]  # Remove (0, 0) pairs (background)

        N = pairs.shape[0]
        instance_masks = np.zeros((H, W, N), dtype=bool)
        instance_labels = np.zeros(N, dtype=int)

        # Generate instance masks and labels
        for i, (label, instance_id) in enumerate(pairs):
            instance_masks[:, :, i] = (img_object_labels == label) & (img_instances == instance_id)
            instance_labels[i] = label

        return instance_masks, instance_labels

    def _get_bounding_boxes(self, instance_map, label_map):
        """
        Extracts bounding boxes and class labels using get_instance_masks.
        Args:
            label_map (numpy.ndarray): 2D array (H, W) with class labels for each pixel.
            instance_map (numpy.ndarray): 2D array (H, W) with instance IDs for each pixel.
        Returns:
            bboxes (list): List of bounding boxes in the format [x_center, y_center, width, height, class_label].
        """
        # Use get_instance_masks to extract binary masks and instance labels
        instance_masks, instance_labels = self._get_instance_masks(label_map, instance_map)

        bboxes = []
        for i in range(instance_masks.shape[-1]):
            mask = instance_masks[:, :, i]  # Binary mask for the current instance
            class_label = instance_labels[i] - 1  # Corresponding class label

            # Find the bounding box from the mask
            y, x = np.where(mask)  # Get mask coordinates
            x_min, x_max = x.min(), x.max()
            y_min, y_max = y.min(), y.max()

            # Normalize bounding box coordinates
            H, W = mask.shape
            x_center = (x_min + x_max) / 2 / W
            y_center = (y_min + y_max) / 2 / H
            width = (x_max - x_min) / W
            height = (y_max - y_min) / H

            # If the bounding box does not meet the requirement of the YOLO model, skip it
            if x_center <= 0 or y_center <= 0 or width <= 0 or height <= 0 or x_center >= 1 or y_center >= 1 or width >= 1 or height >= 1:
                continue

            # Append the bounding box and class label
            bboxes.append([x_center, y_center, width, height, class_label])

        return bboxes

In [6]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from matplotlib import colors
from matplotlib.patches import Rectangle
from matplotlib.colors import ListedColormap

def plot_bounding_boxes(image, bboxes):
    """Visualizes bounding boxes on the image."""
    print("image.shape", image.shape)
    image = np.transpose(image, (1, 2, 0))  # Convert to (H, W, C) format
    fig, ax = plt.subplots(1, figsize=(12, 8))
    ax.imshow(image)

    height, width, _ = image.shape
    for bbox in bboxes:
        x_center, y_center, w, h, class_label = bbox
        x_min = (x_center - w / 2) * width
        y_min = (y_center - h / 2) * height
        box_width = w * width
        box_height = h * height

        rect = patches.Rectangle(
            (x_min, y_min), box_width, box_height, linewidth=2, edgecolor="red", facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(
            x_min,
            y_min,
            NYU_LABELS[class_label - 1],
            color="white",
            fontsize=10,
            bbox=dict(facecolor="red", alpha=0.5),
        )

    plt.title("Bounding Boxes Overlay")
    plt.show()


def test():
    anchors = ANCHORS
    dataset = NYUYoloDataset(
        mat_file=NYU_PATH,
        image_size=416,
        anchors=anchors,
        S=[13, 26, 52],
        C=NYU_LABELS.__len__(),
        transform=train_transforms,
    )

    S = [13, 26, 52]
    scaled_anchors = torch.tensor(anchors) / (
        1 / torch.tensor(S).unsqueeze(1).unsqueeze(1).repeat(1, 3, 2)
    )
    index = 3
    loader = DataLoader(
        dataset=dataset,
        batch_size=BATCH_SIZE,
        num_workers=0,
        pin_memory=PIN_MEMORY,
        shuffle=True,
        drop_last=False,
    )
    for x, y in loader:
        boxes = []
        for i in range(y[0].shape[1]):
            anchor = scaled_anchors[i]
            boxes += cells_to_bboxes(
                y[i], is_preds=False, S=y[i].shape[2], anchors=anchor
            )[0]
        boxes = non_max_suppression(boxes, iou_threshold=1,
                    threshold=0.7, box_format="midpoint")
        plot_image(x[0].permute(1, 2, 0).to("cpu"), boxes)
        break

# test()

## loss.py

In [7]:
# loss.py

import random
import torch
import torch.nn as nn


class YoloLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.mse = nn.MSELoss()
        self.bce = nn.BCEWithLogitsLoss()
        self.entropy = nn.CrossEntropyLoss()
        self.sigmoid = nn.Sigmoid()

        self.lambda_class = 1
        self.lambda_noobj = 10
        self.lambda_obj = 1
        self.lambda_box = 10

    def forward(self, predictions, target, anchors):
        """
        predicitions: (batch_size, num_anchors_on_scale, grid_size, grid_size, 5 + num_classes)
        target: (batch_size, num_anchors_on_scale, grid_size, grid_size, 6)
        anchors: anchor boxes of shape (anchors_on_scale, 2) on a particular scale
        """

        obj = target[..., 0] == 1
        noobj = target[..., 0] == 0

        # NO OBJECT LOSS
        no_object_loss = self.bce(
            predictions[..., 0:1][noobj], target[..., 0:1][noobj])

        # OBJECT LOSS

        '''
        See, per scale, there are three anchor boxes, each being denoted by a x and y coordinate. 
        So, 3 boxes, each having 2 values
        Hence, reshape to 3 x 2.
        To support broadcasting, unsqueezing this wherever necessary.
        '''
        anchors = anchors.reshape(1, 3, 1, 1, 2)

        box_preds = torch.cat([self.sigmoid(predictions[..., 1:3]), torch.exp(
            predictions[..., 3:5]) * anchors], dim=-1)

        ious = intersection_over_union(
            box_preds[obj], target[..., 1:5][obj]).detach()

        object_loss = self.bce(
            (predictions[..., 0:1][obj]), (ious * target[..., 0:1][obj]))

        # BOX COORDINATES

        predictions[..., 1:3] = self.sigmoid(predictions[..., 1:3])
        target[..., 3:5] = torch.log((1e-16 + target[..., 3:5] / anchors))

        box_loss = self.mse(predictions[..., 1:5][obj], target[..., 1:5][obj])

        # CLASS LOSS

        class_loss = self.entropy(
            predictions[..., 5:][obj], target[..., 5][obj].long())

        return (
            self.lambda_box * box_loss +
            self.lambda_obj * object_loss +
            self.lambda_noobj * no_object_loss +
            self.lambda_class * class_loss
        )

## train.py


In [8]:
# train.py

import torch
import torch.optim as optim
import os

from tqdm import tqdm


def train_fn(train_loader, model, optimizer, loss_fn, scaled_anchors):
    loop = tqdm(train_loader, leave=True)
    losses = []

    for batch_idx, (x, y) in enumerate(loop):
        x = x.to(DEVICE)
        y0, y1, y2 = (
            y[0].to(DEVICE),
            y[1].to(DEVICE),
            y[2].to(DEVICE)
        )

        out = model(x)
        loss = (
            loss_fn(out[0], y0, scaled_anchors[0]) +
            loss_fn(out[1], y1, scaled_anchors[1]) +
            loss_fn(out[2], y2, scaled_anchors[2])
        )


        losses.append(loss.item())
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        mean_loss = sum(losses) / len(losses)
        loop.set_postfix({
            "batch_loss": loss.item(),
            "mean_loss": mean_loss,
        })


def main():
    model = YOLOv3(num_classes=NUM_CLASSES).to(DEVICE)
    optimizer = optim.Adam(
        model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY
    )
    loss_fn = YoloLoss()

    train_loader, test_loader, train_eval_loader = get_loaders(
        train_csv_path=DATASET + "/train.csv", test_csv_path=DATASET + "/test.csv"
    ) if not DATASET == 'NYUv2' else get_loaders_nyu(
        mat_file_path=NYU_PATH
    )

    if LOAD_MODEL:
        load_checkpoint(
            CHECKPOINT_FILE, model, optimizer, LEARNING_RATE
        )

    # Scale anchors to each prediction scale
    scaled_anchors = (
        torch.tensor(ANCHORS)
        * torch.tensor(S).unsqueeze(1).unsqueeze(1).repeat(1, 3, 2)
    ).to(DEVICE)

    for epoch in range(NUM_EPOCHS):
        print(f"Epoch [{epoch + 1}/{NUM_EPOCHS}]")
        train_fn(train_loader, model, optimizer, loss_fn, scaled_anchors)

        if SAVE_MODEL:
            save_checkpoint(model, optimizer, filename=f"checkpoint.pth.tar")

        if epoch % 10 == 0 and epoch > 0:
            print("On Test loader:")
            check_class_accuracy(model, test_loader,
                                 threshold=CONF_THRESHOLD)
            # Run model on test set and convert outputs to bounding boxes relative to image
            pred_boxes, true_boxes = get_evaluation_bboxes(
                test_loader,
                model,
                iou_threshold=NMS_IOU_THRESH,
                anchors=ANCHORS,
                threshold=CONF_THRESHOLD,
            )
            # Compute mean average precision
            mapval = mean_average_precision(
                pred_boxes,
                true_boxes,
                iou_threshold=MAP_IOU_THRESH,
                box_format="midpoint",
                num_classes=NUM_CLASSES,
            )
            print(f"MAP: {mapval.item()}")

In [ ]:
main()

Epoch [1/100]


100%|██████████| 145/145 [06:10<00:00,  2.55s/it, batch_loss=48.6, mean_loss=54.8]


=> Saving checkpoint
Epoch [2/100]


 70%|██████▉   | 101/145 [04:07<01:52,  2.55s/it, batch_loss=44.4, mean_loss=46.1]